# Fraud Compliance Agent Notebook 08 — Fast-path model training and evaluation

**Fraud Compliance Agent · Post-Phase-0 candidate evaluation**  
**Status:** Implemented CRISP-DM evaluation harness — synthetic demonstration by default  
**Decision supported:** Candidate model-release review only; never an autonomous payment decision

---

## In plain English

This notebook is the **practice ground for the quick first-pass model**. It shows how several candidate models would be trained and compared using an agreed dataset, while measuring useful trade-offs such as catching risky cases versus wrongly flagging legitimate ones.

The default run uses made-up data to demonstrate the mechanics safely. It is like a driving simulator: it lets us test the dashboard and evaluation process, but it is not evidence that the model can drive on real roads. Results here never make a payment decision or release a model into the app.

## Table of contents

1. [Business Understanding](#business-understanding)
2. [Data Understanding](#data-understanding)
3. [Data Preparation](#data-preparation)
4. [Modelling](#modelling)
5. [Evaluation](#evaluation)
6. [Deployment](#deployment)
7. [References](#7-references)

### How to use this notebook

- **`synthetic`** is the default: it renders the full demonstration with generated fixtures. Every metric and chart is visibly labelled mechanics-only, never fraud-model evidence.
- **`gate`** is available through the FCA_NOTEBOOK08_MODE environment variable when you want to confirm that real training remains blocked.
- **`approved`** is available only after an accepted, checksum-verified local corpus and model-training contract exist. It produces candidate evidence, never a runtime release.

This is a CRISP-DM notebook, not a production scoring service. The API remains the only eventual runtime home for approved, tested logic.




<a id="business-understanding"></a>
## 1. Business Understanding

### 1.1 Decision to support

The business question is: **does a proposed tabular model improve the fast-path risk estimate for an approved target, without compromising policy controls or customer outcomes?**

The model may estimate risk. It cannot approve, challenge, hold, release, or execute a payment. Deterministic fraud and APP controls, authority, oversight, and review remain independent and precede any action.

### 1.2 Success criteria and constraints

Success is not a single accuracy number. A candidate must be assessed against approved prevalence, PR-AUC, ROC-AUC, calibration, precision, recall, false-positive rate, cohort slices, operational review capacity, and fraud/false-decline cost assumptions.

The target, corpus, feature set, chronological partitions, calibration procedure, and release criteria must be accepted before real training. A model cannot stand in for an APP control if its target does not represent APP risk.

### 1.3 Run context and reproducibility

The next cell records the repository revision and selects a controlled execution mode. It does not load data or train a model.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import tempfile
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


def find_repository_root(start: Path) -> Path:
    """Return the monorepo root so file paths never depend on a notebook UI cwd."""
    for candidate in (start, *start.parents):
        if (candidate / "docs" / "project-context.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the fraud-compliance-agent repository.")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
DEFAULT_MODE = "synthetic"
MODE = os.getenv("FCA_NOTEBOOK08_MODE", DEFAULT_MODE).strip().lower()
MANIFEST_PATH = REPOSITORY_ROOT / "docs" / "contracts" / "model-training-contract.v1.json"
REVIEWED_REPORT_PATH = REPOSITORY_ROOT / "docs" / "proposals" / "fast-path-model-release.candidate.json"
# Only an approved-mode run may replace the reviewed report. Gate and synthetic
# runs default to a temporary file so a direct notebook run cannot overwrite it.
DEFAULT_REPORT_PATH = (
    REVIEWED_REPORT_PATH
    if MODE == "approved"
    else Path(tempfile.gettempdir()) / f"fast-path-model-release.{MODE}.json"
)
REPORT_PATH = Path(os.getenv("FCA_NOTEBOOK08_REPORT_PATH", str(DEFAULT_REPORT_PATH))).resolve()
RANDOM_SEED = 20260916

if MODE not in {"gate", "synthetic", "approved"}:
    raise RuntimeError("FCA_NOTEBOOK08_MODE must be gate, synthetic, or approved.")


def git_revision() -> str:
    """Return the current Git revision without exposing command errors in output."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "uncommitted-or-unavailable"


RUN_CONTEXT = {
    "run_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": git_revision(),
    "mode": MODE,
    "random_seed": RANDOM_SEED,
}
print("Repository:", REPOSITORY_ROOT)
print("Mode:", MODE)
print("Safety: no raw data, identifiers, model weights, or release decision is written to Git.")





<a id="data-understanding"></a>
## 2. Data Understanding

### 2.1 Required evidence

Approved mode requires a versioned model-training contract that declares the local dataset path and checksum, target, feature columns, precomputed partitions, calibration procedure, slices, and release-criteria version. This prevents a notebook user from silently changing the question, labels, or data.

### 2.2 What we know today

The accepted contract currently authorises a Sparkov **mechanics-only** benchmark, not a production corpus or target. Plaid Sandbox observations remain integration evidence, not labelled fraud data. Approved mode consumes only the contract-declared, checksum-verified local dataset; it never fetches, manufactures, or silently substitutes data. Any results must remain labelled synthetic benchmark mechanics, not fraud-model performance evidence.

The next cell enforces those boundaries before any dataset is considered.



In [ ]:
REQUIRED_ACCEPTED_FIELDS = {
    "approval_status",
    "dataset_path",
    "dataset_sha256",
    "target_column",
    "feature_columns",
    "partition_column",
    "calibration_partition",
    "calibration_procedure",
    "test_partition",
    "slice_columns",
    "release_criteria_version",
}


def load_accepted_manifest() -> dict[str, object]:
    """Load only a complete, explicitly accepted training contract."""
    if not MANIFEST_PATH.is_file():
        raise RuntimeError(
            "Missing accepted model-training contract at docs/contracts/model-training-contract.v1.json. "
            "Create and approve it through the contract/ADR process; do not infer values here."
        )
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    missing = sorted(REQUIRED_ACCEPTED_FIELDS - set(manifest))
    if missing or manifest.get("approval_status") != "accepted":
        raise RuntimeError(
            "Model-training contract is not accepted or is incomplete: " + ", ".join(missing or ["approval_status"])
        )
    return manifest


if MODE == "gate":
    print("GATED — real model training is disabled by default.")
    print("Set FCA_NOTEBOOK08_MODE=approved only after the accepted contract, corpus, feature schema, partitions, calibration procedure, and release criteria exist.")
    print("Set FCA_NOTEBOOK08_MODE=synthetic only to test notebook mechanics; it cannot support a performance claim.")
    manifest: dict[str, object] | None = None
elif MODE == "approved":
    manifest = load_accepted_manifest()
    print("Accepted training contract present; validating only its declared local dataset path and checksum next.")
    print("Training scope:", manifest.get("training_scope", "not declared"))
else:
    manifest = None
    print("SYNTHETIC DEMONSTRATION MODE — the full workflow and visual diagnostics will render.")
    print("Labels are generated solely to exercise code and cannot be used as fraud-model evidence.")



<a id="data-preparation"></a>
## 3. Data Preparation

### 3.1 Prepare an analysis-ready, point-in-time dataset

This stage validates only the contract-declared schema. It checks the dataset checksum, required columns, target, and distinct train/calibration/test partitions. It does not repair missing fields, impute a target, resample classes, or invent unavailable signals.

### 3.2 Leakage and feature-parity guardrails

Real inputs must use the accepted point-in-time feature contract from Notebooks 04–07. A feature is eligible only if it can be computed from facts available before decision time and has an online-equivalent calculation. Chronological partitions are mandatory; a random split is not substituted here.

The next cell loads either the explicit synthetic fixture or the accepted local CSV and validates those boundaries.


In [ ]:
def synthetic_mechanics_dataset() -> pd.DataFrame:
    """Synthetic fixture for mechanics only: not a fraud corpus and never a model claim."""
    rng = np.random.default_rng(RANDOM_SEED)
    size = 900
    timestamp = pd.date_range("2025-01-01", periods=size, freq="h", tz="UTC")
    amount = rng.lognormal(mean=4.3, sigma=1.0, size=size)
    velocity_6h = rng.poisson(lam=1.8, size=size)
    account_age_days = rng.integers(1, 1200, size=size)
    is_new_payee = rng.integers(0, 2, size=size)
    # This target is deliberately synthetic/rule-shaped. It validates the
    # mechanics only and must never be interpreted as fraud-performance evidence.
    synthetic_probability = 1 / (1 + np.exp(-(-6 + 0.010 * amount + 0.42 * velocity_6h + 1.1 * is_new_payee - 0.001 * account_age_days)))
    target = rng.binomial(1, np.clip(synthetic_probability, 0.001, 0.999))
    partition = np.where(np.arange(size) < 540, "train", np.where(np.arange(size) < 720, "calibration", "test"))
    return pd.DataFrame({
        "event_time": timestamp,
        "amount_minor": np.round(amount * 100).astype(int),
        "velocity_6h": velocity_6h,
        "account_age_days": account_age_days,
        "is_new_payee": is_new_payee,
        "target": target,
        "partition": partition,
        "synthetic_slice": np.where(is_new_payee == 1, "new_payee", "known_payee"),
    })


def approved_dataset(manifest: dict[str, object]) -> tuple[pd.DataFrame, dict[str, object]]:
    """Load the contract-declared local CSV after verifying its immutable checksum."""
    dataset_path = Path(str(manifest["dataset_path"])).expanduser()
    if not dataset_path.is_absolute():
        dataset_path = REPOSITORY_ROOT / dataset_path
    if not dataset_path.is_file():
        raise RuntimeError("Approved dataset path is unavailable locally. Do not replace it with another dataset.")
    if dataset_path.suffix.lower() != ".csv":
        raise RuntimeError("This notebook accepts approved local CSV input only. Extend the accepted contract before using another format.")
    digest = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
    if digest != manifest["dataset_sha256"]:
        raise RuntimeError("Dataset checksum does not match the accepted training contract.")
    return pd.read_csv(dataset_path), manifest


if MODE == "synthetic":
    frame = synthetic_mechanics_dataset()
    contract = {
        "target_column": "target",
        "feature_columns": ["amount_minor", "velocity_6h", "account_age_days", "is_new_payee"],
        "partition_column": "partition",
        "calibration_partition": "calibration",
        "calibration_procedure": "synthetic-diagnostic-only",
        "test_partition": "test",
        "slice_columns": ["synthetic_slice"],
        "dataset_sha256": "synthetic-mechanics-only",
        "release_criteria_version": "not-applicable",
    }
elif MODE == "approved":
    frame, contract = approved_dataset(manifest)
else:
    frame, contract = None, None

if frame is not None:
    required_columns = set(contract["feature_columns"]) | {
        contract["target_column"], contract["partition_column"], *contract["slice_columns"]
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise RuntimeError("Approved/synthetic data is missing declared columns: " + ", ".join(missing_columns))
    partitions = set(frame[contract["partition_column"]].dropna().astype(str))
    required_partitions = {"train", str(contract["calibration_partition"]), str(contract["test_partition"])}
    if not required_partitions <= partitions:
        raise RuntimeError("Declared train/calibration/test partitions are incomplete.")
    print("Dataset schema validated. Rows:", len(frame))
    print("Feature count:", len(contract["feature_columns"]))


<a id="modelling"></a>
## 4. Modelling

### 4.1 Baseline and candidate

The class-balanced logistic-regression pipeline is the interpretable baseline. XGBoost is the proposed nonlinear tabular-model comparison, not an approved production algorithm. Its hyperparameters and imbalance weight are fixed for a reproducible comparison only; they must be reviewed under the accepted evaluation protocol. Both models use the same declared feature columns and deterministic seed.

### 4.2 Training protocol

Models fit only the train partition. The held-out test partition is never used to fit either model. The separate calibration partition is required by the contract for the approved calibration procedure; this notebook will not invent one.

The next cell fits the baseline and XGBoost comparison pipelines and produces scores only when data is explicitly available in synthetic or approved mode.



In [ ]:
def build_models(y_train: pd.Series) -> dict[str, Pipeline]:
    """Return a baseline and XGBoost comparison model with no action authority."""
    positive_count = int(y_train.sum())
    negative_count = len(y_train) - positive_count
    if positive_count == 0 or negative_count == 0:
        raise RuntimeError("The train partition must contain both target classes.")

    # This weight addresses class imbalance for a fixed comparison only. Its
    # value and all hyperparameters must be reviewed under the accepted protocol.
    scale_pos_weight = negative_count / positive_count
    return {
        "logistic_regression_baseline": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED)),
        ]),
        "xgboost_candidate": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                n_jobs=1,
                random_state=RANDOM_SEED,
                tree_method="hist",
            )),
        ]),
    }


def predict_and_evaluate(model: Pipeline, x_train: pd.DataFrame, y_train: pd.Series, x_test: pd.DataFrame, y_test: pd.Series) -> tuple[np.ndarray, dict[str, float]]:
    """Fit on the train partition and return held-out scores plus aggregate metrics."""
    model.fit(x_train, y_train)
    scores = model.predict_proba(x_test)[:, 1]
    metrics = {
        "pr_auc": float(average_precision_score(y_test, scores)),
        "roc_auc": float(roc_auc_score(y_test, scores)),
        "brier_score": float(brier_score_loss(y_test, scores)),
    }
    return scores, metrics


if frame is not None:
    target = str(contract["target_column"])
    partition = str(contract["partition_column"])
    feature_columns = list(contract["feature_columns"])
    train_frame = frame.loc[frame[partition] == "train"].copy()
    calibration_frame = frame.loc[frame[partition] == contract["calibration_partition"]].copy()
    test_frame = frame.loc[frame[partition] == contract["test_partition"]].copy()
    if train_frame[target].nunique() != 2 or calibration_frame[target].nunique() != 2 or test_frame[target].nunique() != 2:
        raise RuntimeError("Every declared partition must contain both target classes; do not resample silently.")
    model_scores: dict[str, np.ndarray] = {}
    model_metrics: dict[str, dict[str, float]] = {}
    for name, model in build_models(train_frame[target]).items():
        scores, metrics = predict_and_evaluate(
            model,
            train_frame[feature_columns], train_frame[target],
            test_frame[feature_columns], test_frame[target],
        )
        model_scores[name] = scores
        model_metrics[name] = metrics
    print("Models evaluated:", ", ".join(model_metrics))
    print("Metrics are", "synthetic mechanics only." if MODE == "synthetic" else "candidate evidence pending review.")



<a id="evaluation"></a>
## 5. Evaluation

### 5.1 Model quality versus business policy

PR-AUC, ROC-AUC, and Brier score describe score quality. A threshold sweep then shows the policy trade-off between precision, recall, false-positive rate, and block rate. No threshold is selected here: that is an independent policy, authority, and oversight decision.

### 5.2 Slice and limitation review

Aggregate performance can hide harm or operational regressions in a subgroup. The notebook therefore reports approved slice metrics where both classes are present. Synthetic results remain a software test only; they cannot establish fairness, fraud efficacy, or release readiness.

The next cell produces only aggregate diagnostics and never a payment action.


In [ ]:
def threshold_sweep(y_true: pd.Series, scores: np.ndarray) -> list[dict[str, float]]:
    """Describe score-policy trade-offs without selecting an operating threshold."""
    rows: list[dict[str, float]] = []
    for threshold in np.arange(0.05, 1.00, 0.05):
        prediction = scores >= threshold
        negatives = y_true == 0
        false_positive_rate = float(((prediction == 1) & negatives).sum() / negatives.sum()) if negatives.sum() else float("nan")
        rows.append({
            "threshold": round(float(threshold), 2),
            "precision": float(precision_score(y_true, prediction, zero_division=0)),
            "recall": float(recall_score(y_true, prediction, zero_division=0)),
            "false_positive_rate": false_positive_rate,
            "block_rate": float(prediction.mean()),
        })
    return rows


def slice_metrics(frame: pd.DataFrame, slice_column: str, y_true: pd.Series, scores: np.ndarray) -> list[dict[str, object]]:
    """Compute aggregate diagnostics per approved slice, skipping one-class slices."""
    result: list[dict[str, object]] = []
    for value, indices in frame.groupby(slice_column, dropna=False).groups.items():
        y_slice = y_true.loc[indices]
        score_slice = scores[list(indices)]
        if y_slice.nunique() != 2:
            continue
        result.append({
            "slice": slice_column,
            "value": str(value),
            "count": len(indices),
            "prevalence": float(y_slice.mean()),
            "pr_auc": float(average_precision_score(y_slice, score_slice)),
            "roc_auc": float(roc_auc_score(y_slice, score_slice)),
        })
    return result


if frame is not None:
    # No threshold is selected here. This is a report of trade-offs for a later,
    # independent policy decision; the model has no authority to act.
    test_target = test_frame[target].reset_index(drop=True)
    test_for_slices = test_frame.reset_index(drop=True)
    evaluation = {}
    for name, scores in model_scores.items():
        evaluation[name] = {
            "metrics": model_metrics[name],
            "threshold_sweep": threshold_sweep(test_target, scores),
            "slice_metrics": [
                metric
                for slice_column in contract["slice_columns"]
                for metric in slice_metrics(test_for_slices, slice_column, test_target, scores)
            ],
        }
    print("Threshold sweeps and slice metrics calculated; no operating threshold selected.")


### 5.3 Visual diagnostics

The following Plotly diagnostics use the Arbiris SDK notebook presentation convention: a white canvas, bold title, muted subtitle, explicit margins, readable hover details, and the established blue/red/green palette. In synthetic mode, every chart is visibly labelled as mechanics-only and cannot support a fraud-performance claim. In approved mode, it remains a candidate-evaluation view with no promotion or threshold decision.


In [ ]:
# These visual diagnostics use the shared Arbiris notebook convention:
# white canvas, bold title, muted subtitle, clear margins, and explicit hover text.
# They visualise aggregate held-out scores only. They do not select a threshold
# or grant the model authority over payment decisions.
PLOTLY_RENDER = os.getenv("FCA_NOTEBOOK08_RENDER_PLOTS", "true").strip().lower() not in {
    "0", "false", "no",
}
MODEL_COLORS = {
    "logistic_regression_baseline": "#2E91E5",
    "xgboost_candidate": "#EF553B",
}
PLOT_WIDTH = 900
PLOT_MARGIN = {"l": 100, "r": 200, "t": 100, "b": 75}


def evaluation_subtitle(detail: str) -> str:
    """Return a safety-qualified subtitle for every visual diagnostic."""
    mode_notice = (
        "Synthetic mechanics-only — not fraud-model performance evidence."
        if MODE == "synthetic"
        else "Candidate evaluation — review only; no threshold or release decision."
    )
    return f"{detail}<br><sup>{mode_notice}</sup>"


def create_evaluation_figure(title: str, subtitle: str, hovermode: str = "closest") -> go.Figure:
    """Create an Arbiris-style Plotly figure with one reusable visual language."""
    figure = go.Figure()
    figure.update_layout(
        template="plotly_white",
        title=f"<b>{title}</b><br><sup>{subtitle}</sup>",
        title_font_size=18,
        width=PLOT_WIDTH,
        hovermode=hovermode,
        margin=PLOT_MARGIN,
        legend={
            "bgcolor": "rgba(255, 255, 255, 0.85)",
            "bordercolor": "#E5ECF6",
            "borderwidth": 1,
        },
    )
    figure.update_xaxes(showgrid=True, gridcolor="#E5ECF6", ticklabelstandoff=6, zeroline=False)
    figure.update_yaxes(showgrid=True, gridcolor="#E5ECF6", zeroline=False)
    return figure


def reliability_curve(y_true: pd.Series, scores: np.ndarray, bins: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """Return populated equal-width score-bin means for a reliability diagnostic."""
    table = pd.DataFrame({"outcome": y_true.to_numpy(), "score": scores})
    table["bin"] = pd.cut(table["score"], bins=np.linspace(0, 1, bins + 1), include_lowest=True)
    grouped = table.groupby("bin", observed=True).agg(
        mean_prediction=("score", "mean"),
        observed_rate=("outcome", "mean"),
    )
    return grouped["mean_prediction"].to_numpy(), grouped["observed_rate"].to_numpy()


def add_safety_annotation(figure: go.Figure) -> None:
    """Place a visible non-authority notice below a diagnostic chart."""
    figure.add_annotation(
        text="Diagnostic only — no operating threshold is selected.",
        xref="paper",
        yref="paper",
        x=0,
        y=-0.24,
        showarrow=False,
        font={"color": "#6B7280", "size": 11},
        align="left",
    )


visualisations: dict[str, go.Figure] = {}

if frame is None:
    print("Visual diagnostics are gated until synthetic mechanics mode or an accepted training contract supplies data.")
else:
    # Precision-recall curves foreground performance under class imbalance.
    pr_figure = create_evaluation_figure(
        "Precision–recall curves",
        evaluation_subtitle("Held-out test partition; dashed line is test prevalence."),
    )
    pr_figure.add_hline(
        y=float(test_target.mean()),
        line_dash="dot",
        line_color="#6B7280",
        annotation_text="Test prevalence",
        annotation_position="bottom right",
    )
    for name, scores in model_scores.items():
        precision, recall, _ = precision_recall_curve(test_target, scores)
        pr_figure.add_trace(
            go.Scatter(
                x=recall,
                y=precision,
                mode="lines",
                name=name.replace("_", " "),
                line={"color": MODEL_COLORS[name], "width": 3},
                hovertemplate="Recall: %{x:.3f}<br>Precision: %{y:.3f}<extra>%{fullData.name}</extra>",
            )
        )
    pr_figure.update_xaxes(title_text="Recall", range=[0, 1])
    pr_figure.update_yaxes(title_text="Precision", range=[0, 1])
    add_safety_annotation(pr_figure)
    visualisations["precision_recall"] = pr_figure

    # ROC curves remain useful as a complementary discrimination diagnostic.
    roc_figure = create_evaluation_figure(
        "ROC curves",
        evaluation_subtitle("Held-out test partition; dashed diagonal is no-discrimination reference."),
    )
    roc_figure.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            name="No-discrimination reference",
            line={"color": "#6B7280", "dash": "dot"},
            hoverinfo="skip",
        )
    )
    for name, scores in model_scores.items():
        false_positive_rate, true_positive_rate, _ = roc_curve(test_target, scores)
        roc_figure.add_trace(
            go.Scatter(
                x=false_positive_rate,
                y=true_positive_rate,
                mode="lines",
                name=name.replace("_", " "),
                line={"color": MODEL_COLORS[name], "width": 3},
                hovertemplate="False-positive rate: %{x:.3f}<br>True-positive rate: %{y:.3f}<extra>%{fullData.name}</extra>",
            )
        )
    roc_figure.update_xaxes(title_text="False-positive rate", range=[0, 1])
    roc_figure.update_yaxes(title_text="True-positive rate", range=[0, 1])
    add_safety_annotation(roc_figure)
    visualisations["roc"] = roc_figure

    # Reliability plots expose probability calibration; they do not calibrate scores.
    calibration_figure = create_evaluation_figure(
        "Reliability diagnostic",
        evaluation_subtitle("Held-out test partition; score bins are descriptive and do not calibrate the model."),
    )
    calibration_figure.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            name="Perfect calibration",
            line={"color": "#6B7280", "dash": "dot"},
            hoverinfo="skip",
        )
    )
    for name, scores in model_scores.items():
        mean_prediction, observed_rate = reliability_curve(test_target, scores)
        calibration_figure.add_trace(
            go.Scatter(
                x=mean_prediction,
                y=observed_rate,
                mode="lines+markers",
                name=name.replace("_", " "),
                line={"color": MODEL_COLORS[name], "width": 3},
                marker={"size": 8},
                hovertemplate="Mean predicted score: %{x:.3f}<br>Observed rate: %{y:.3f}<extra>%{fullData.name}</extra>",
            )
        )
    calibration_figure.update_xaxes(title_text="Mean predicted score", range=[0, 1])
    calibration_figure.update_yaxes(title_text="Observed outcome rate", range=[0, 1])
    add_safety_annotation(calibration_figure)
    visualisations["reliability"] = calibration_figure

    # Overlaid score distributions make class separation legible without a policy cut-off.
    distribution_figure = create_evaluation_figure(
        "Held-out score distributions",
        evaluation_subtitle("Outcome groups in the test partition; opacity is used so overlap remains visible."),
    )
    distribution_figure.update_layout(barmode="overlay")
    for name, scores in model_scores.items():
        for outcome, color, label in (
            (0, "#2E91E5", "Observed non-target"),
            (1, "#EF553B", "Observed target"),
        ):
            distribution_figure.add_trace(
                go.Histogram(
                    x=scores[test_target.to_numpy() == outcome],
                    nbinsx=30,
                    name=f"{name.replace('_', ' ')} — {label}",
                    legendgroup=name,
                    marker={"color": color},
                    opacity=0.38 if name == "logistic_regression_baseline" else 0.65,
                    hovertemplate="Score: %{x:.3f}<br>Count: %{y}<extra>%{fullData.name}</extra>",
                )
            )
    distribution_figure.update_xaxes(title_text="Model score", range=[0, 1])
    distribution_figure.update_yaxes(title_text="Count")
    add_safety_annotation(distribution_figure)
    visualisations["score_distribution"] = distribution_figure

    # The trade-off chart reports, but never chooses, a possible policy threshold.
    threshold_figure = create_evaluation_figure(
        "Threshold trade-offs",
        evaluation_subtitle("Held-out test partition; comparison only — policy must select no threshold here."),
        hovermode="x unified",
    )
    measure_styles = {
        "precision": ("Precision", "#2E91E5"),
        "recall": ("Recall", "#00CC96"),
        "false_positive_rate": ("False-positive rate", "#EF553B"),
        "block_rate": ("Score-at-or-above threshold", "#636EFA"),
    }
    for name, values in evaluation.items():
        sweep = pd.DataFrame(values["threshold_sweep"])
        for metric, (label, color) in measure_styles.items():
            threshold_figure.add_trace(
                go.Scatter(
                    x=sweep["threshold"],
                    y=sweep[metric],
                    mode="lines+markers",
                    name=f"{name.replace('_', ' ')} — {label}",
                    legendgroup=name,
                    line={"color": color, "width": 3, "dash": "solid" if name == "logistic_regression_baseline" else "dash"},
                    marker={"size": 6},
                    hovertemplate="Threshold: %{x:.2f}<br>Rate: %{y:.3f}<extra>%{fullData.name}</extra>",
                )
            )
    threshold_figure.update_xaxes(title_text="Candidate threshold (not selected)", range=[0, 1])
    threshold_figure.update_yaxes(title_text="Rate", range=[0, 1])
    add_safety_annotation(threshold_figure)
    visualisations["threshold_tradeoffs"] = threshold_figure

    print("Created visual diagnostics:", ", ".join(visualisations))
    if PLOTLY_RENDER:
        for figure in visualisations.values():
            figure.show()
    else:
        print("Plot rendering disabled for this run; set FCA_NOTEBOOK08_RENDER_PLOTS=true to display figures.")




<a id="deployment"></a>
## 6. Deployment

### 6.1 Candidate-release evidence, not deployment

In CRISP-DM, deployment here means placing a reviewable result into the delivery process. The report records aggregate metrics, configuration identity, partition counts, limitations, and a digest. It does not save model weights to Git, expose raw data, choose a policy threshold, or change API behaviour.

### 6.2 Required next steps before any runtime use

An independently approved release process must verify artifact storage/checksum, online–offline feature parity, deterministic-control precedence, contract version, monitoring and drift plan, rollback plan, and an authorised human decision. Only then may reusable implementation move into tested API code.


In [ ]:
if MODE == "gate":
    report = {
        "artifact": "fast-path-model-release",
        "status": "gated",
        "run_context": RUN_CONTEXT,
        "blocking_requirements": [
            "accepted model-training contract", "approved corpus and target", "approved feature contract",
            "chronological partitions", "calibration procedure", "release criteria",
        ],
        "decision_recommendation": "proposed — do not train or promote a candidate model",
    }
else:
    report = {
        "artifact": "fast-path-model-release",
        "status": "synthetic_mechanics_only" if MODE == "synthetic" else "candidate_evaluation_pending_review",
        "run_context": RUN_CONTEXT,
        "input_manifest": {
            "dataset_sha256": contract["dataset_sha256"],
            "feature_columns": contract["feature_columns"],
            "target_column": contract["target_column"],
            "release_criteria_version": contract["release_criteria_version"],
        },
        "partition_counts": {
            "train": len(train_frame), "calibration": len(calibration_frame), "test": len(test_frame),
        },
        "prevalence": {
            "train": float(train_frame[target].mean()), "calibration": float(calibration_frame[target].mean()), "test": float(test_frame[target].mean()),
        },
        "models": evaluation,
        "limitations": [
            "No policy threshold was selected; model scores have no payment authority.",
            "Deterministic controls, authority, oversight, and review remain independent runtime controls.",
            "Synthetic mechanics results are not fraud-model performance evidence and cannot support release or promotion.",
        ] if MODE == "synthetic" else [
            "Candidate results are Sparkov synthetic benchmark mechanics only and cannot support a production fraud-performance claim.",
            "Candidate results require independent review against accepted release criteria.",
            "This notebook does not promote a model or select a payment-action threshold.",
        ],
        "decision_recommendation": "proposed — no runtime promotion or policy decision is implied",
    }

payload = json.dumps(report, sort_keys=True, indent=2)
report["report_sha256"] = hashlib.sha256(payload.encode("utf-8")).hexdigest()
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, sort_keys=True, indent=2) + "\n", encoding="utf-8")

report_location = (
    str(REPORT_PATH.relative_to(REPOSITORY_ROOT))
    if REPORT_PATH.is_relative_to(REPOSITORY_ROOT)
    else "temporary external output"
)
print("Sanitised report written:", report_location)
print("Report status:", report["status"])
print("Report SHA-256:", report["report_sha256"])
print("No model weights, source rows, identifiers, or policy threshold were written.")


## 7. References

### Fraud-ML and evaluation

1. Stripe. (2021, December 15). *A primer on machine learning for fraud detection*. Stripe Radar Technical Guide. https://stripe.com/gb/guides/primer-on-machine-learning-for-fraud-protection  
   Used for the separation of model quality from policy thresholds, precision/recall and false-positive trade-offs, held-out evaluation, real-time feature parity, monitoring, and drift.
2. scikit-learn developers. (n.d.). *Model evaluation: quantifying the quality of predictions*. scikit-learn documentation. https://scikit-learn.org/stable/modules/model_evaluation.html  
   Used as the implementation reference for the aggregate evaluation metrics in this mechanics harness.

### Code-style convention

3. van delay. (2024). *Does it really matter what order you import modules in Python?* r/Python discussion. https://www.reddit.com/r/Python/comments/1bv9of6/does_it_really_matter_what_order_you_import/  
   Applied as a convention: standard-library imports, then third-party imports, then project-local imports, with blank lines between groups. Ruff's isort-compatible `I` rules enforce this convention.
4. Astral. (n.d.). *Ruff rule reference: isort (I)*. https://docs.astral.sh/ruff/rules/  
   Project enforcement reference for import ordering.

No other research paper or external fraud-data blog was used to create this notebook. The synthetic fixture exists only to test mechanics and is not a research source.


## Review checklist

- [ ] Confirm the execution mode was appropriate and visible in the report.
- [ ] Confirm approved mode used an accepted contract and checksum-verified local data.
- [ ] Confirm synthetic output is labelled mechanics-only.
- [ ] Review model metrics, calibration procedure, slices, and threshold trade-offs together.
- [ ] Confirm no threshold, release, or payment authority was inferred from this notebook.
- [ ] Clear notebook outputs before commit.
- [ ] Recommendation remains **proposed**: no runtime promotion or policy decision is implied.
